### 🚀 `07_cicd_pipeline.ipynb` – CI/CD Pipeline and Model Monitoring

This notebook brings together everything built in the project to show how a **CI/CD (Continuous Integration and Continuous Deployment)** process works in AWS SageMaker. The main goal is to demonstrate how **model monitoring results** can act as a checkpoint—automatically deciding if a new model version is ready for production or should be reviewed or rejected.

---

### ✅ What This Notebook Does

- **Deploys the Model to an Endpoint**  
  It deploys the trained model to a SageMaker endpoint that captures all input and output data during predictions.

- **Simulates Monitoring Results**  
  Instead of relying on live monitor runs (which were inconsistent in this environment), it simulates what a SageMaker Model Monitor job would return. This allows us to reliably demonstrate the logic behind automatic model approval or rejection.

- **Automates Model Approval**  
  Based on whether the simulated monitor result passes or fails, the notebook updates the model's approval status in the **SageMaker Model Registry**.

- **(Optional) Cleans Up Resources**  
  Includes optional code to delete the deployed endpoint and monitoring schedule to avoid ongoing costs.

---

### 🧠 Why This Matters

In real-world machine learning workflows (MLOps), it's not enough to just train a good model. Once deployed, models can behave differently as the data they see in production changes over time. **Model monitoring** helps detect these issues early.

This notebook shows how to:

- Use model monitoring as a key quality check in your deployment pipeline
- Automatically approve or reject models based on monitoring outcomes
- Simulate a full CI/CD loop for validating and updating ML models

---

### 🔗 How It Builds on Earlier Notebooks

- **Model Training**  
  The model was trained in `03_model_training.ipynb`.

- **Model Packaging and Monitoring Baseline**  
  The model and the inference script were bundled and registered in the **Model Registry** in `05_monitoring_and_registry.ipynb`, which also created the baseline statistics and constraints needed for monitoring.

- **Endpoint Data Capture**  
  In this notebook (`07`), the endpoint is configured to record all prediction data so it can be analyzed by the monitor.

---

### ⚠️ A Note About Monitoring Challenges

During testing, SageMaker Model Monitor jobs failed to run consistently. Some of the issues included format mismatches (e.g., expecting NumPy but getting CSV), and generic container errors with no logs in CloudWatch. These problems seemed to be caused by **limitations in the lab environment**, not by errors in the project code.

To work around this, the notebook **simulates monitor results** instead. This allows the CI/CD logic to be demonstrated fully, including how model approval decisions are made—even when the monitor doesn’t run successfully.


## Sagemaker Setup

In [75]:
import boto3
from sagemaker import get_execution_role
from sagemaker.session import Session

# Session and role setup
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_session = Session(boto_session=boto_session)
sagemaker_client = boto_session.client("sagemaker")
role = get_execution_role()

# S3 bucket and endpoint
bucket = sagemaker_session.default_bucket()
endpoint_name = "readmission-endpoint18"


## Deploy Model

In [76]:
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.sklearn.model import SKLearnModel

model_artifact_uri = f"s3://{bucket}/diabetes/registry/model.tar.gz"

sklearn_model = SKLearnModel(
    model_data=model_artifact_uri,
    role=role,
    entry_point="inference.py",
    framework_version="1.0-1",
    sagemaker_session=sagemaker_session
)

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f"s3://{bucket}/monitoring/data_capture",
    capture_options=["Input", "Output"]
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config
)

print("Model deployed to endpoint:", endpoint_name)


------!Model deployed to endpoint: readmission-endpoint18


## Test Deployment

In [77]:
import pandas as pd
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

X_val = pd.read_pickle("data/X_val.pkl")

# Select a single row and ensure integer types
row = X_val.iloc[0].astype(int)

# Configure predictor for CSV input and output
predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

# Make prediction
response = predictor.predict(row.tolist())

print("Response:", response)


Response: [['0']]


## Monitoring in CI/CD

### A) Hourly Monitor 

In [78]:
monitor_schedule_name = "readmission-monitor-schedule"
baseline_s3_uri = f"s3://{bucket}/monitoring/baseline"

In [79]:
print(f"Attempting to delete existing monitoring schedule '{monitor_schedule_name}'...")
try:
    sagemaker_client.delete_monitoring_schedule(MonitoringScheduleName=monitor_schedule_name)
    print(f"Existing monitoring schedule '{monitor_schedule_name}' deleted. Waiting for deletion to complete...")
    # Give AWS a moment to process the deletion
    time.sleep(10) # Small delay to ensure deletion propagates
except ClientError as e:
    if "ValidationException" in str(e) and "Could not find monitoring schedule" in str(e):
        print(f"Monitoring schedule '{monitor_schedule_name}' does not exist, proceeding to create.")
    elif "ResourceNotFoundException" in str(e): # Sometimes this error is raised instead of ValidationException for not found
        print(f"Monitoring schedule '{monitor_schedule_name}' not found, proceeding to create.")
    else:
        print(f"Error deleting monitoring schedule '{monitor_schedule_name}': {e}")
except Exception as e:
    print(f"An unexpected error occurred during schedule deletion: {e}")

Attempting to delete existing monitoring schedule 'readmission-monitor-schedule'...
Existing monitoring schedule 'readmission-monitor-schedule' deleted. Waiting for deletion to complete...


In [80]:
# Hourly Monitor

from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator, EndpointInput
from botocore.exceptions import ClientError
import time 

monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

endpoint_input = EndpointInput(
    endpoint_name=endpoint_name,
    destination="/opt/ml/processing/input",
)

print(f"Creating new hourly monitoring schedule '{monitor_schedule_name}' for endpoint '{endpoint_name}'...")

monitor.create_monitoring_schedule(
    monitor_schedule_name=monitor_schedule_name,
    endpoint_input=endpoint_input,
    output_s3_uri=f"s3://{bucket}/monitoring/reports",
    statistics=baseline_s3_uri + "/statistics.json",
    constraints=baseline_s3_uri + "/constraints.json",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True
)

print("Monitoring schedule created successfully.")

Creating new hourly monitoring schedule 'readmission-monitor-schedule' for endpoint 'readmission-endpoint18'...
Monitoring schedule created successfully.


In [72]:
import boto3
from botocore.exceptions import ClientError
import time # For polling if needed

def check_monitor_status(schedule_name):
    # Checks and prints the status of the latest monitoring job execution
    try:
        response = sagemaker_client.describe_monitoring_schedule(MonitoringScheduleName=schedule_name)
        status = response.get("LastMonitoringExecutionSummary", {}).get("MonitoringExecutionStatus", "Unknown")
        print(f"Monitor Status: {status}")
        return status
    except ClientError as e:
        print(f"Error checking monitor status for {schedule_name}: {e}")
        return "ERROR"

# Call the function to check status
monitor_schedule_name = 'readmission-monitor-schedule'
monitor_status = check_monitor_status(monitor_schedule_name)

print("Waiting for monitor job to complete...")
while monitor_status not in ["Completed", "CompletedWithViolations", "Failed", "ERROR"]:
    time.sleep(60) # Wait 60 seconds before re-checking
    monitor_status = check_monitor_status(monitor_schedule_name)
print(f"Monitor job finished with status: {monitor_status}")

Monitor Status: Failed
Waiting for monitor job to complete...
Monitor job finished with status: Failed


In [81]:
import pandas as pd
import time # Import time for the delay
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

# Load validation data (assuming 'data/X_val.pkl' is accessible)
X_val = pd.read_pickle("data/X_val.pkl")

# Configure predictor for CSV input and output
# These lines must be run after your endpoint is deployed and 'predictor' object is available
predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

print("Sending multiple inferences to populate data capture for monitoring...")

# Define how many inferences to send. A few hundred is usually sufficient for monitoring.
# You can adjust this number based on how much data your monitor needs.
num_inferences_to_send = 50 # Start with 50, increase if monitor still reports too few records

# Iterate through X_val and send each row as a prediction request
# We'll limit it to 'num_inferences_to_send' rows or the total size of X_val, whichever is smaller.
for i in range(min(num_inferences_to_send, len(X_val))):
    # Select a row and ensure integer types as required by your model
    row_to_predict = X_val.iloc[i].astype(int).tolist()

    try:
        # Make prediction
        response = predictor.predict(row_to_predict)
        # print(f"Inference {i+1} response: {response}") # Uncomment if you want to see each response

        # Add a small delay between requests to simulate realistic traffic and avoid throttling
        # and to give data capture time to write
        time.sleep(0.1) # 100 milliseconds delay

        if (i + 1) % 10 == 0: # Print update every 10 inferences
            print(f"  Sent {i+1} inferences...")

    except Exception as e:
        print(f"Error sending inference {i+1}: {e}")
        print("Stopping further inferences due to error.")
        break

print(f"Finished sending {min(num_inferences_to_send, len(X_val))} sample inferences to endpoint '{endpoint_name}'.")
print("Data capture will now accumulate in your S3 bucket.")


Sending multiple inferences to populate data capture for monitoring...
  Sent 10 inferences...
  Sent 20 inferences...
  Sent 30 inferences...
  Sent 40 inferences...
  Sent 50 inferences...
Finished sending 50 sample inferences to endpoint 'readmission-endpoint18'.
Data capture will now accumulate in your S3 bucket.


### B) On Demand Monitor

In [83]:
import boto3
import sagemaker
import time
from botocore.exceptions import ClientError

# --- Configuration for the On-Demand Monitor Job ---
job_name_prefix = "manual-monitor-run"
timestamp = time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
processing_job_name = f"{job_name_prefix}-{timestamp}"

# Baseline S3 URI (from 05_monitoring_and_registry.ipynb)
baseline_s3_uri = f"s3://{bucket}/monitoring/baseline"
# S3 path where this specific job's reports will be stored
output_s3_uri_job = f"s3://{bucket}/monitoring/manual_reports/{timestamp}"

# Clarify/Model Monitor image URI
# This retrieves the correct image for SageMaker Clarify in your region
clarify_image_uri = sagemaker.image_uris.retrieve(
    framework='clarify',
    region=region,
    version='latest' # Use 'latest' or a specific version like '1.0' or '2.0'
)

print(f"Launching on-demand Model Monitor Processing Job: {processing_job_name}")
print(f"Clarify Image URI: {clarify_image_uri}")
print(f"Output S3 URI: {output_s3_uri_job}")

try:
    response = sagemaker_client.create_processing_job(
        ProcessingJobName=processing_job_name,
        ProcessingResources={
            'ClusterConfig': {
                'InstanceCount': 1,
                'InstanceType': 'ml.m5.large', # Match your monitor schedule instance type
                'VolumeSizeInGB': 20 # Match your monitor schedule volume size
            }
        },
        AppSpecification={
            'ImageUri': clarify_image_uri,
            'ContainerArguments': [
                # Arguments for the SageMaker Clarify container to run model monitoring
                # These are crucial for the monitor to know what to do.
                '--monitoring_type', 'DataQuality', # Or 'ModelQuality', 'ModelBias', 'ModelExplainability'
                '--dataset_format', 'text/csv', # Your captured data format
                '--endpoint_input', endpoint_name,
                '--output_s3_uri', output_s3_uri_job,
                '--baseline_s3_uri', baseline_s3_uri,
                # Optional: specify output content type for endpoint (to match inference.py)
                # This is sometimes handled by the monitor internally, but can be explicit if needed
                # '--accept_type', 'application/x-npy' # Match what your inference.py outputs for monitor
            ],
            'ContainerEntrypoint': ['/opt/ml/processing/run_monitor'] # Standard entrypoint for clarify monitor
        },
        RoleArn=role,
        ProcessingInputs=[
            {
                'InputName': 'baseline',
                'S3Input': {
                    'S3Uri': baseline_s3_uri,
                    'LocalPath': '/opt/ml/processing/input/baseline',
                    'S3DataType': 'S3Prefix', # Specifies it's a prefix containing multiple files
                    'S3InputMode': 'File',
                    'S3DataDistributionType': 'FullyReplicated'
                }
            },
            {
                'InputName': 'endpoint_input', # This name is internal to how clarify expects input
                'S3Input': {
                    'S3Uri': f"s3://{bucket}/monitoring/data_capture/{endpoint_name}/", # Data capture from your endpoint
                    'LocalPath': '/opt/ml/processing/input/endpoint_input',
                    'S3DataType': 'S3Prefix', # Data capture comes as a prefix of hourly/daily folders
                    'S3InputMode': 'File',
                    'S3DataDistributionType': 'FullyReplicated'
                }
            }
        ],
        ProcessingOutputConfig={
            'Outputs': [
                {
                    'OutputName': 'monitoring_output',
                    'S3Output': {
                        'S3Uri': output_s3_uri_job,
                        'LocalPath': '/opt/ml/processing/output',
                        'S3UploadMode': 'EndOfJob'
                    }
                }
            ]
        },
        StoppingCondition={
            'MaxRuntimeInSeconds': 1800 # 30 minutes
        },
        Tags=[{'Key': 'MonitorType', 'Value': 'OnDemand'}] # Optional tag for easier filtering
    )

    print("\nOn-demand monitoring job launched successfully!")
    print(f"Job Name: {processing_job_name}")
    print(f"Job ARN: {response['ProcessingJobArn']}")

    # --- Polling for status (optional, but useful for immediate feedback) ---
    print("\nPolling job status (check CloudWatch logs for details)...")
    while True:
        job_desc = sagemaker_client.describe_processing_job(ProcessingJobName=processing_job_name)
        status = job_desc['ProcessingJobStatus']
        print(f"Job Status: {status} (Last Update: {job_desc['LastModifiedTime'].strftime('%H:%M:%S')})")

        if status in ['Completed', 'Failed', 'Stopped']:
            break
        time.sleep(30) # Check every 30 seconds

    print(f"\nOn-demand monitoring job finished with status: {status}")
    if status == 'Failed':
        print(f"Failure Reason: {job_desc.get('FailureReason', 'N/A')}")
        print("Check CloudWatch logs for detailed error: ")
        print(f"https://console.aws.amazon.com/cloudwatch/home?region={region}#logStream:group=/aws/sagemaker/ProcessingJobs/{processing_job_name}")


except ClientError as e:
    print(f"Error launching on-demand monitoring job: {e}")
    print("Ensure all S3 paths, IAM role, and instance types are correct.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Launching on-demand Model Monitor Processing Job: manual-monitor-run-2025-06-25-02-50-44
Clarify Image URI: 205585389593.dkr.ecr.us-east-1.amazonaws.com/sagemaker-clarify-processing:1.0
Output S3 URI: s3://sagemaker-us-east-1-380537322556/monitoring/manual_reports/2025-06-25-02-50-44

On-demand monitoring job launched successfully!
Job Name: manual-monitor-run-2025-06-25-02-50-44
Job ARN: arn:aws:sagemaker:us-east-1:380537322556:processing-job/manual-monitor-run-2025-06-25-02-50-44

Polling job status (check CloudWatch logs for details)...
Job Status: InProgress (Last Update: 02:50:44)
Job Status: InProgress (Last Update: 02:50:45)
Job Status: InProgress (Last Update: 02:50:45)
Job Status: InProgress (Last Update: 02:50:45)
Job Status: InProgress (Last Update: 02:50:45)
Job Status: InProgress (Last Update: 02:50:45)
Job Status: InProgress (Last Update: 02:53:16)
Job Status: InProgress (Last Update: 02:53:16)
Job Status: InProgress (Last Update: 02:53:16)
Job Status: InProgress (Last Up

## Simulated Monitor Output

### Summary: Challenges with SageMaker Model Monitor Execution

Throughout the development of the CI/CD pipeline, the **SageMaker Model Monitor** component (`DefaultModelMonitor`) encountered persistent execution issues, preventing it from completing successfully and providing a direct signal for automated model approval.

#### Key Issues Observed

---

**1. Initial Encoding Mismatch**  
*Error:* `AlgorithmError: Encoding is CSV for endpointInput, but Encoding is BASE64 for endpointOutput`

- **Problem:** There was a mismatch in expected data formats. The monitor correctly sent `text/csv` to the endpoint's `input_fn`, but unexpectedly expected `application/x-npy` from the `output_fn`. However, the deployed `inference.py` was returning `text/csv`.
  
- **Root Cause:** This discrepancy was traced to an older version of the SageMaker Python SDK in the lab environment, which did not allow explicit configuration of the monitor’s expected `accept_type` (i.e., `accept_attribute` on `EndpointInput` or `accept_type` in `DefaultModelMonitor`).

- **Resolution Attempt:** The `inference.py` script was updated to dynamically support both `text/csv` (for user inputs) and `application/x-npy` (for monitor requests) within its `output_fn`.

---

**2. Subsequent Generic `AlgorithmError` with Missing CloudWatch Logs**  
*Error:* `Algorithm container exited with error. Please try again.`

- **Problem:** Even after fixing the format handling, both scheduled and manually triggered monitor jobs continued to fail with a generic `AlgorithmError`.  
- **Diagnostics Challenge:** No detailed Python traceback was found in CloudWatch. In some cases, CloudWatch log groups were entirely missing for the job.

- **Likely Cause:** These symptoms strongly suggest that the monitoring container **crashed silently or was terminated prematurely**, likely due to:
  - Resource limitations
  - Timeout constraints in the lab environment
  - Instability in the runtime infrastructure

- **Impact:** The abrupt container exit prevented logs from flushing to CloudWatch, making the root cause opaque.

---

#### Impact on CI/CD Demonstration

Although SageMaker Model Monitor could not be executed to completion due to environmental instability, the CI/CD pipeline’s logic for **model approval/rejection** based on monitor outcomes was implemented and demonstrated effectively.

- By **simulating monitor results** (`Completed` or `CompletedWithViolations`), the system's ability to **automatically approve or reject models** in the **SageMaker Model Registry** was validated.
- This showcases the **critical role of Model Monitor** as a gatekeeper in MLOps workflows—even when operating under constrained or imperfect conditions.


In [84]:
print("--- SIMULATING MODEL MONITOR OUTCOME FOR DEMO PURPOSES ---")
print("In a real CI/CD pipeline, this outcome would be derived automatically")
print("from the execution status and violation reports of the SageMaker Model Monitor job.")

# Scenario: Simulate a SUCCESSFUL monitoring run (monitor passes, no violations)
simulated_monitor_status = "Completed"
print(f"\nSimulating scenario: Monitor Status is '{simulated_monitor_status}' (No violations detected).")

model_approved = False # Default to not approved

if simulated_monitor_status == "Completed":
    print("CI/CD Decision: Model monitoring passed without violations. Proceeding to Model Approval.")
    model_approved = True
elif simulated_monitor_status == "CompletedWithViolations":
    print("CI/CD Decision: Model monitoring completed with violations. Model will be flagged for review/rejection.")
    # For this demo, we'll auto-reject models with violations.
    model_approved = False
elif simulated_monitor_status == "Failed":
    print(f"CI/CD Decision: Model monitoring failed with status: {simulated_monitor_status}. Model will be rejected.")
    model_approved = False
else:
    # This case handles 'Unknown' or other unexpected statuses
    print(f"CI/CD Decision: Unexpected monitor status: {simulated_monitor_status}. Model not approved.")
    model_approved = False


--- SIMULATING MODEL MONITOR OUTCOME FOR DEMO PURPOSES ---
In a real CI/CD pipeline, this outcome would be derived automatically
from the execution status and violation reports of the SageMaker Model Monitor job.

Simulating scenario: Monitor Status is 'Completed' (No violations detected).
CI/CD Decision: Model monitoring passed without violations. Proceeding to Model Approval.


## Model Approval

In [85]:
# From 05
model_package_arn = "arn:aws:sagemaker:us-east-1:380537322556:model-package/ReadmissionModelGroup/20" 

if model_approved:
    print("\nAttempting to APPROVE model in Model Registry...")
    try:
        response = sagemaker_client.update_model_package(
            ModelPackageArn=model_package_arn,
            ModelApprovalStatus="Approved"
        )
        print(f"Model package APPROVED: {model_package_arn}")
    except Exception as e:
        print(f"Error approving model package: {e}")
        print("Ensure the model package ARN is correct and the IAM role has 'sagemaker:UpdateModelPackage' permission.")
else:
    print("\nAttempting to REJECT model in Model Registry...")
    try:
        response = sagemaker_client.update_model_package(
            ModelPackageArn=model_package_arn,
            ModelApprovalStatus="Rejected" # You can also use "PendingManualApproval" for review
        )
        print(f"Model package REJECTED: {model_package_arn}. Manual review might be required.")
    except Exception as e:
        print(f"Error rejecting model package: {e}")
        print("Ensure the model package ARN is correct and the IAM role has 'sagemaker:UpdateModelPackage' permission.")



Attempting to APPROVE model in Model Registry...
Model package APPROVED: arn:aws:sagemaker:us-east-1:380537322556:model-package/ReadmissionModelGroup/20


## Endpoint Deletion

Uncomment and run following cell to delete endpoint

In [88]:
# Initialize SageMaker client
sm_client = boto3.client("sagemaker")

# List of endpoint names you want to delete
endpoints_to_delete = [
    "readmission-endpoint18"
]

# Delete related monitoring schedules
schedules = sm_client.list_monitoring_schedules(MaxResults=100)

for sched in schedules['MonitoringScheduleSummaries']:
    endpoint_name = sched['EndpointName']
    sched_name = sched['MonitoringScheduleName']
    if endpoint_name in endpoints_to_delete:
        try:
            sm_client.delete_monitoring_schedule(MonitoringScheduleName=sched_name)
            print(f"Deleted monitoring schedule: {sched_name}")
        except Exception as e:
            print(f"Error deleting monitoring schedule {sched_name}: {e}")

# Delete the endpoints
for ep in endpoints_to_delete:
    try:
        sm_client.delete_endpoint(EndpointName=ep)
        print(f"Deleted endpoint: {ep}")
    except Exception as e:
        print(f"Error deleting endpoint {ep}: {e}")


Deleted endpoint: readmission-endpoint18
